# שיעור 11 - פרוטוקול סוכן-לסוכן (A2A)


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## מהו פרוטוקול A2A?

ה**פרוטוקול סוכן-לסוכן (A2A)** הוא תקן פתוח שמאפשר לסוכני בינה מלאכותית לתקשר,
לגלות זה את זה, ולשתף פעולה — גם כשהם נבנים על גבי מסגרות שונות או מתארחים
על ידי שירותים שונים.

מושגי מפתח:

- **גילוי** – סוכנים מפרסמים *כרטיס סוכן* שמתאר את יכולותיהם, מה שהופך
  קל לסוכנים אחרים (או אורקסטרטורים) למצוא את המומחה הנכון למשימה.
- **העברת הודעות** – סוכנים מחליפים הודעות מובנות דרך פרוטוקול משותף, כך ש
  בקשה של סוכן אחד יכולה להיות מובנת ומבוצעת על ידי אחר ללא תלות ב
  המימוש הפנימי שלה.
- **מחזור חיי המשימה** – A2A מגדיר מצבים כמו *submitted*, *working*, *completed*, ו
  *failed*, ומעניק לאורקסטרטור נראות מלאה לגבי אופן התקדמות משימה שהופקדה.

בשיעור זה אנו מדמים שיתוף פעולה בסגנון A2A על ידי חיבור של שלושה סוכני נסיעות מתמחים לתוך תהליך עבודה שבו כל סוכן תורם מהמומחיות שלו ומעביר תוצאות לאחר.


## יצירת סוכני נסיעות מתמחים


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## שיתוף פעולה בין סוכנים באמצעות זרימת עבודה

אנו מחברים את שלושת הסוכנים לזרימת עבודה סדרתית שמשקפת את העברת ההודעות A2A:

1. **CurrencyExchangeAgent** מקבל את בקשת המשתמש ומספק הנחיות להמרת מטבע.
2. **ActivityPlannerAgent** מקבל את ההקשר המועשר ומוסיף המלצות לפעילויות.
3. **TravelManagerAgent** משלב את שני הקלטים לתדריך נסיעות סופי.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## הבנת A2A בסביבת ייצור

בסביבת ייצור פרוטוקול A2A פותח תרחישים רבי-עוצמה חוצי-שירותים:

| Capability | Description |
|---|---|
| **אינטרופרביליות בין מסגרות** | סוכן שנבנה עם מסגרת אחת יכול להאציל משימות לסוכן שנבנה עם כל מסגרת אחרת שתואמת ל-A2A, ובכך לאפשר אינטראופרביליות אמיתית בין ארגונים. |
| **גבולות שירות** | סוכנים יכולים להתארח במיקרו-שירותים נפרדים, באזורי ענן נפרדים, ואפילו בארגונים שונים ועדיין לשתף פעולה בצורה חלקה. |
| **גילוי דינמי** | אורקסטרטור יכול לשאול רישום של Agent Card בזמן ריצה כדי למצוא את המומחה המתאים ביותר למשימת משנה נתונה. |
| **שידור & הודעות דחיפה** | A2A תומך ב-Server-Sent Events (SSE) לעדכוני התקדמות בזמן אמת ולהודעות דחיפה עבור משימות ארוכות. |

תהליך העבודה שבנינו למעלה הוא גרסה מפושטת, בתוך התהליך של דפוס זה. בפריסה אמיתית
כל סוכן היה חושף נקודת קצה HTTP, מפרסם Agent Card, ומתקשר
באמצעות פרוטוקול A2A JSON-RPC.


## סיכום

בשיעור זה למדת:

1. **מהו פרוטוקול A2A** — תקן פתוח לגילוי בין סוכנים, לשליחת הודעות ולניהול משימות.
2. **כיצד ליצור סוכנים מתמחים** — סוכן להמרת מטבעות, סוכן לתכנון פעילויות, וסוכן אורקסטרטור לניהול נסיעות.
3. **כיצד לקשר סוכנים לתוך תהליך עבודה** — שימוש ב-`WorkflowBuilder` ליצירת דגם של העברת הודעות סדרתית בין סוכנים.
4. **כיצד A2A פועל בסביבת ייצור** — מאפשר שיתוף פעולה חוצה-מסגרות וחוצה-שירותים עם גילוי דינמי ועדכונים זורמים.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
הצהרת אחריות:
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית Co-op Translator (https://github.com/Azure/co-op-translator). אף שאנו שואפים לדייק, יש לשים לב שתרגומים אוטומטיים עלולים להכיל שגיאות או אי‑דיוקים. יש להסתמך על המסמך המקורי בשפתו כמקור הסמכות. עבור מידע קריטי מומלץ להיעזר בתרגום מקצועי על ידי מתרגם אנושי. איננו נושאים באחריות לכל אי‑הבנות או פרשנויות שגויות הנובעות משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
